# MESSAGEix-Pakistan 
### NDC Conditional Scenario
In this notebook, we are reading data and building a conditional NDC scenerio.

<img src="https://wit.lums.edu.pk/sites/default/files/inline-images/WIT_Banner.jpg" alt="Girl in a jacket" width="850" height="250">

In [ ]:
import os
import sys
from pathlib import Path

# resolve the message_pak root (one level above this notebook's directory)
ROOT = Path(os.path.abspath('')).parent
sys.path.insert(0, str(ROOT))

# fundamental libraries
import ixmp
import message_ix
from message_ix import log

from scripts.calibrate_t_d import model_calibrate
from scripts.reserve_margin import update_reserve_margin
from scripts.emissions import bound_emissions

# autoreload modules when changes are applied to them
%load_ext autoreload  
%autoreload all
%reload_ext autoreload
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')


In [2]:
# Confirm root path (message_pak directory)
print('Project root:', ROOT)

Project root: d:\COMMITTED\Models\Model-Manuscript\NEST-Pakistan\message_pak


Create the Scenario

In [3]:
# creating ixmp platform object
new_mp = ixmp.Platform("committed 1.0")

# creating a new, empty scenario object
scenario = message_ix.Scenario(
    new_mp, 
    model="MESSAGEix-Pakistan", 
    scenario="ndc-c", version="new"
)

Read Model Data

In [ ]:
data_path = ROOT / 'data' / 'MESSAGEix-Pakistan-NDC-C_NZ-SSP2.xlsx'
scenario.read_excel(str(data_path), add_units=True, commit_steps=False, init_items=True)

In [ ]:
update_reserve_margin(scenario, ["R12_PAK"], contin=0.15)

Constrain Emissions

In [6]:
scenario.check_out()
trajectories = ROOT / 'data' / 'emissionAllocation.xlsx'
bound_emissions(scenario, "R12_PAK", "NDC-C", trajectories)

Solve the Model

In [7]:
log.info(f"version number before commit(): {scenario.version}")
scenario.commit("Applied emission constraints for NDC-C scenario")
scenario.set_as_default()

# exporting the built model (Scenario) to GAMS with an optional case name
caseName = scenario.model + '__' + scenario.scenario + '__v' + str(scenario.version)

# solve model
scenario.solve(case=caseName)
obj_value = scenario.var("OBJ")["lvl"]
print(f"Objective value: {obj_value}")

Objective value: 32399.47265625


##### Report the Results

In [8]:
from report.legacy.iamc_report_hackathon import report
from datetime import datetime
import time

timestamp = datetime.now().strftime('%Y-%m-%d--%H-%M')
start = time.time()
out_dir = ROOT / 'output'
out_dir.mkdir(parents=True, exist_ok=True)
df, path_name = report(mp=new_mp, scen=scenario, out_dir=str(out_dir), out_file_timestamp=timestamp, IDEA_format=False)
end = time.time()
print(f'Reporting done in {end - start:.1f}s → {out_dir}')


processing Table: Resource|Extraction
processing Table: Resource|Cumulative Extraction
processing Table: Primary Energy
processing Table: Primary Energy (substitution method)
processing Table: Final Energy
processing Table: Secondary Energy|Electricity
processing Table: Secondary Energy|Heat
processing Table: Secondary Energy
processing Table: Secondary Energy|Gases
processing Table: Secondary Energy|Solids
processing Table: Emissions|CO2
processing Table: Carbon Sequestration
processing Table: Emissions|BC
processing Table: Emissions|OC
processing Table: Emissions|CO
processing Table: Emissions|N2O
processing Table: Emissions|CH4
processing Table: Emissions|NH3
processing Table: Emissions|Sulfur
processing Table: Emissions|NOx
processing Table: Emissions|VOC
processing Table: Emissions|HFC
HFC125
No objects to concatenate
processing Table: Emissions
processing Table: Emissions
processing Table: Agricultural Demand
module 'report.legacy.default_tables_static' has no attribute 'retr_agr

In [9]:
# close the connection to the database
new_mp.close_db()